In [1]:
import pandas as pd
import numpy as np
from datetime import datetime
import boto3
import io
import os
import warnings
import gc
warnings.filterwarnings("ignore")

# Chargement de la base 

In [3]:
def get_s3_client():
    """Client S3/MinIO"""
    return boto3.client(
        "s3",
        endpoint_url="http://192.168.1.230.:30137",
        aws_access_key_id="wer1Or8j7hXa3QGk2M3M",
        aws_secret_access_key="YtbyBd3Y0bEYeDy8aeX3lAf4JIUlKYlxY8lNVTVt",
        verify=False
    )


def charger_panel_solde_1_2():
    """Charge les données depuis S3/MinIO"""
    bucket_name = "admindataanstat"
    object_key = "Solde/Panel_solde_complet_2015_2025.parquet"

    s3_client = get_s3_client()
    obj = s3_client.get_object(Bucket=bucket_name, Key=object_key)
    df = pd.read_parquet(io.BytesIO(obj["Body"].read()))
    
    print(f"✓ Données chargées : {len(df):,} observations")
    if "MATRICULE" in df.columns:
        print(f"✓ Matricules uniques : {df['MATRICULE'].nunique():,}")
    if "DATE_COLLECTE" in df.columns:
        print(f"✓ Période : {df['DATE_COLLECTE'].min()} → {df['DATE_COLLECTE'].max()}")
    
    return df


In [4]:
panel = charger_panel_solde_1_2()


EndpointConnectionError: Could not connect to the endpoint URL: "http://192.168.1.230.:30137/admindataanstat/Solde/Panel_solde_complet_2015_2025.parquet"

In [ ]:
panel.columns.tolist()


In [ ]:
panel["age"].describe()
panel["sexe"].value_counts()
panel.info()


In [ ]:
df["age"].unique()


# Calcul des indicateurs 

In [ ]:
import pandas as pd
import numpy as np

MOIS_FR = {
    1: "Janvier", 2: "Février", 3: "Mars", 4: "Avril", 5: "Mai", 6: "Juin",
    7: "Juillet", 8: "Août", 9: "Septembre", 10: "Octobre", 11: "Novembre", 12: "Décembre"
}

def build_indicateurs_salaire(
    panel,
    out_xlsx_path="indicateurs_salaire_2015_2025.xlsx",
    out_csv_path="indicateurs_salaire_2015_2025.csv"
):
    df = panel.copy()

    # Colonnes (figées selon ta demande)
    ANNEE_COL = "ANNEE_COLLECTE" if "ANNEE_COLLECTE" in df.columns else "YEAR"
    MOIS_COL = "MOIS_COLLECTE" if "MOIS_COLLECTE" in df.columns else "MONTH"
    BRUT_COL = "MONTANT BRUT_B1" if "MONTANT BRUT_B1" in df.columns else "MONTANT BRUT_B0"
    NET_COL = "MONTANT NET"
    SEXE_COL = "SEXE_IMPUTE"
    CAT_COL = "CATEGORIE_1"
    MAT_COL = "MATRICULE"

    # Sécurisation types
    df[NET_COL] = pd.to_numeric(df[NET_COL], errors="coerce")
    df[BRUT_COL] = pd.to_numeric(df[BRUT_COL], errors="coerce")

    # Année / mois
    df["ANNEE"] = pd.to_numeric(df[ANNEE_COL], errors="coerce")
    df["MOIS_NUM"] = pd.to_numeric(df[MOIS_COL], errors="coerce")
    df["MOIS"] = df["MOIS_NUM"].map(MOIS_FR)

    # Sexe
    df["SEXE_STD"] = df[SEXE_COL].astype(str).str.upper().map({
        "M": "Homme", "H": "Homme", "HOMME": "Homme",
        "F": "Femme", "FEMME": "Femme"
    })

    # Statut
    df["STATUT"] = np.where(
        df[MAT_COL].astype(str).str.startswith(("5", "6")),
        "Non fonctionnaire",
        "Fonctionnaire"
    )

    # Fonction d’agrégation
    def agg(g):
        return pd.Series({
            "Salaire mensuel net moyen": g[NET_COL].mean(),
            "Salaire mensuel brut moyen": g[BRUT_COL].mean(),
            "Salaire médian mensuel net": g[NET_COL].median(),
            "Salaire médian mensuel brut": g[BRUT_COL].median(),
            "Min net": g[NET_COL].min(),
            "Max net": g[NET_COL].max(),
        })

    key = ["ANNEE", "MOIS_NUM", "MOIS"]
    frames = {}

    # Fonctionnaires par catégorie
    for cat in sorted(df[CAT_COL].dropna().unique()):
        frames[f"Fonctionnaire - Cat {cat}"] = (
            df[(df["STATUT"] == "Fonctionnaire") & (df[CAT_COL] == cat)]
            .groupby(key)
            .apply(agg)
        )

    # Non fonctionnaires
    frames["Non fonctionnaire"] = (
        df[df["STATUT"] == "Non fonctionnaire"]
        .groupby(key)
        .apply(agg)
    )

    # Ensemble
    frames["Ensemble"] = df.groupby(key).apply(agg)

    # Sexe
    frames["Homme"] = df[df["SEXE_STD"] == "Homme"].groupby(key).apply(agg)
    frames["Femme"] = df[df["SEXE_STD"] == "Femme"].groupby(key).apply(agg)

    # Tableau final
    result = pd.concat(frames, axis=1).reset_index()
    result = result.sort_values(["ANNEE", "MOIS_NUM"]).drop(columns="MOIS_NUM")

    # Exports
    result.to_excel(out_xlsx_path, index=False)
    result.to_csv(out_csv_path, index=False, encoding="utf-8-sig")

    return result


In [ ]:
# Charger les données
panel = charger_panel_solde_1_2()

# Lancer l'indicateur
table_indicateurs = build_indicateurs_salaire(
    panel,
    out_xlsx_path="indicateurs_salaire_2015_2025.xlsx",
    out_csv_path="indicateurs_salaire_2015_2025.csv"
)

table_indicateurs.head()
    